# DataONE Replication Demo

This notebook demonstrates how to use the DataONE replication service to replicate data from a DataONE Member repository (MN) to another member repository. It uses the DataONE Python libraries to interact with the DataONE APIs.
The DataONE python libraries are located in GitHub at [github.com/DataONEorg/d1_python](https://github.com/DataONEorg/d1_python) with documentation at [dataone-python.readthedocs.io](https://dataone-python.readthedocs.io/en/latest/). We also provide the [DataONE R client packages](https://dataoneorg.r-universe.dev/dataone) for R users (also downloadable from [CRAN](https://cran.r-project.org/web/packages/dataone/index.html).

## Overview of replication

![DataONE Replication](images/dataone-replication.png)

## Installation

Installation under a virtual environment is recommended. [`uv`](https://docs.astral.sh/uv/) is a tool for creating and managing virtual environments in Python. It is similar to `virtualenv` but has some additional features. You can use any virtual environment tool that allows you to load the following packages, including the dependencies for a Jupyter notebook. We'll be demonstrating using Visual Studio Code with the Jupyter extension, but you can use any Jupyter notebook environment.

uv:

```bash
mkdir d1-replication-demo
cd d1-replication-demo
uv init
uv add dataone.cli 
uv add ipykernel
```

While you could create your own virtual environment and notebook, we have created a template to work from that includes functions that will be helpful during the demo. Clone or download the repository from https://github.com/DataONEorg/d1-replication-demo. Once you have a copy of the repository, change your working directory and install the dependencies:

```bash
cd d1-replication-demo
uv install
```

## Create a dataset for testing

Let's create a dataset for testing replication in a DataONE testing environment, particularly using the [KNB Test Repository](https://dev.nceas.ucsb.edu/). 

![KNB](images/knb-banner.png)

The dataset will consist of a single CSV file and the metadata describing it, but could in theory contain as many objects as needed.

### Login to the repository using your ORCID identity

DataONE uses ORCID as a common user identifier for sharing across the network of repositories. If you do not have an ORCID account, you can create one at [orcid.org](https://orcid.org/).

Click the 'Sign In' button in the top right corner of the KNB Test Repository page, and then click the 'Sign in with ORCID' button. You will be redirected to the ORCID login page. After logging in, you will be redirected back to the KNB Test Repository.

![KNB Sign In](images/knb-login.png)

### Save your DataONE API key

Later we will want to use the DataONE API to interact with the repository. To do this, we need an API key that is associated with your ORCID account. We'll copy the API key to a file named `jwtkey` in the current directory. This key will be used to authenticate your requests to the DataONE API when we are working in python. You can find the API key in the KNB Test Repository by clicking on your name in the top right corner, then selecting 'My profile'. In the 'Session' tab, you will see your API key listed under 'Authentication Token'. Copy this key and save it to a file named `jwtkey` in the current directory.

![KNB API Key](images/knb-auth-token.png)

### Create a new dataset 

Now let's create a simple single file dataset. Click 'Submit' to start a new dataset, and then pick a small CSV file to upload (we provide one in the example repository `data` directory if you don't have one handy). You can also add a PNG image to the dataset, but this is not required. 

![KNB Submit](images/knb-submit.png)

Once you have added your files, you'll need to provide the minimal metadata required to create the dataset. This includes a title, abstract, and name and email for the creator and contact. You can also add additional metadata if you wish, but this is not required for the demo. You can find the required metadata fields as they are marked with a red asterisk (*).

![KNB Dataset](images/knb-dataset.png)

# Replication across DataONE

Now that we have a dataset in the KNB Test Repository, we can replicate it to another DataONE Member Node (MN). The replication service allows us to create copies of the dataset in other repositories, which can be useful for backup and redundancy purposes. Each object in DataONE is referenced via its Persistent Identifier (PID), which is a unique identifier for the object and visible in the citation on the dataset landing page. The PID is used to retrieve the object from the repository, and it is also used to set the replication policy for the object.

## Getting a repository client

Let's start by importing the necessary libraries and setting up the DataONE client to interact with the KNB Test Repository. We will use the `d1_client` library to create a client for the KNB Test Repository, which is a member repository in the test network.

The DataONE python clients provide an abstraction of the DataONE APIs enabling interaction with DataONE Member (MN) and Coordinating (CN) Nodes. Since the APIs of MNs and CNs differ, there are two basic types of client - one for Member repositories (MNs) and one for Coordinating Nodes (CNs) that derive from a common base.

In [1]:
import datetime
import d1_client.mnclient_2_0
import d1_client.cnclient_2_0
import d1_common.types.exceptions


In [2]:
# Authentication token (JWT) from the DataONE Test Environment
# Create a file named jwtkey with the token on the first line and no other lines
with open("jwtkey", "r") as f:
    for line in f:
        auth_token = line.strip()

# Set the token in the request header
options: dict = {"headers": {"Authorization": "Bearer " + auth_token}}

# Base URL of the KNB Test Repository
base_url = "https://dev.nceas.ucsb.edu/knb/d1/mn"
repo = d1_client.mnclient_2_0.MemberNodeClient_2_0(base_url, **options)

# Call the getCapabilities API method
# https://dataone-architecture-documentation.readthedocs.io/en/latest/apis/MN_APIs.html#MNCore.getCapabilities
node = repo.getCapabilities()

# Response is an instance of a Node document that can be accessed through its properties
# https://dataone-architecture-documentation.readthedocs.io/en/latest/apis/Types.html#Types.Node
print(f"{node.name:30} {node.baseURL}")


KNB Test Node                  https://dev.nceas.ucsb.edu/knb/d1/mn


Note that the variable `repo` represents the KNB Test Repository, and we can use that variable to manipulate our dataset. 

## Getting a CN client

In addition to the repository, we can also interact with the DataONE Coordinating Nodes (CNs), which provide network-wide services that are useful to clients. The CNs provide services such as replication, synchronization, and search. We can create a client for the CNs using the `d1_cn_client` library, and then use that to verify that our saved API key is recognized and valid. You should see your ORCID credentials printed out in the output of the cell below.

In [3]:
# The Staging CN base URL:
base_url = "https://cn-stage.test.dataone.org/cn"
cn = d1_client.cnclient_2_0.CoordinatingNodeClient_2_0(base_url, **options)

creds = cn.echoCredentials()
print(creds.toxml())


<?xml version="1.0" ?><ns1:subjectInfo xmlns:ns1="http://ns.dataone.org/service/types/v1"><person><subject>http://orcid.org/0000-0003-0077-4738</subject><givenName>Matthew B.</givenName><familyName>Jones</familyName><isMemberOf>CN=DBO,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=SASAP,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=delta-swg-23,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=SASAP Data Team,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=Likely Suspects Framework Users,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=drp-portal-admins,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=DataONE-Support,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=drp-data-admins,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=cib-admins,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=vegbank-admins,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=vegbank-contributors,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=cib-curators,DC=dataone,DC=org</isMemberOf><isMemberOf>CN=knb-data-admins,DC=dataone,DC=org</isMemberOf><i

## Functions for viewing system metadata

Each object in DataONE is identified by its PID and is associated with system metadata that describes the object, including its size, format, and replication policy. The system metadata is stored in the repository and can be retrieved using the DataONE API. Let's create a `view` function to print system metadata for a given PID. This function will be used to view the system metadata for our dataset and any replicated objects.
 

In [4]:
def toStr(v):
    if isinstance(v, datetime.datetime):
        return v.strftime(DATE_FORMAT)
    return str(v)

def propertyStr(p):
    '''String representation of pyxb property
    '''
    if p is None:
        return ""
    try:
        return toStr(p.value())
    except:
        return toStr(p)

def view(sysmeta, title="SystemMetadata"):
    '''Print SystemMetadata properties
    '''
    print(f"\n{title}:")
    print(f"  Identifier: {sysmeta.identifier.value()}")
    print(f"  Series Identifier: {propertyStr(sysmeta.seriesId)}")
    print(f"  Modified: {sysmeta.dateSysMetadataModified}")
    print(f"  Uploaded: {sysmeta.dateUploaded}")
    print(f"  Format ID: {sysmeta.formatId}")
    print(f"  Size: {sysmeta.size}")
    print(f"  Checksum: hash://{sysmeta.checksum.algorithm.lower()}/{sysmeta.checksum.value()}")
    print(f"  Origin Member Node: {propertyStr(sysmeta.originMemberNode)}")
    print(f"  Authoritative Member Node: {propertyStr(sysmeta.authoritativeMemberNode)}")
    print(f"  Obsoletes: {propertyStr(sysmeta.obsoletes)}")
    print(f"  Obsoleted By: {propertyStr(sysmeta.obsoletedBy)}")
    print("  Access policy rules:")
    for rule in sysmeta.accessPolicy.allow:
        print(f"    {', '.join(map(lambda S: S.value(), rule.subject))}  can  {', '.join(rule.permission)}")
    print("  Replication policy:")
    print(f"    Replication allowed: {sysmeta.replicationPolicy.replicationAllowed}")
    print(f"    Replicas requested: {sysmeta.replicationPolicy.numberReplicas}")
    for node in sysmeta.replicationPolicy.preferredMemberNode:
        print(f"    Preferred node: {node.value()}")
    for node in sysmeta.replicationPolicy.blockedMemberNode:
        print(f"    Blocked node: {node.value()}")
    print("  Replicas of this object:")
    for replica in sysmeta.replica:
        print(f"    {replica.replicaMemberNode.value():15} {replica.replicationStatus:10} {replica.replicaVerified}")

## Function to change a replication policy

Changing the replication policy allows setting how many replicas of an object are requested, which nodes are preferred for replication, and which nodes are blocked from replication. The replication policy is part of the SystemMetadata for an object. We'll create a function to change key aspects of a replication policy, including:

- the number of replicas requested,
- the preferred nodes for replication

The function will also display the metadata before and after the change was made, so you can see the effect of the change.


In [5]:
def change_replication_policy(repo, pid, number_replicas=0, preferred_node=None):
    '''Change the replication policy to allow replication
    '''

    try:
        sysmeta = repo.getSystemMetadata(pid)
        view(sysmeta, "Original SystemMetadata")
        if number_replicas > 0:
            sysmeta.replicationPolicy.replicationAllowed = True
            sysmeta.replicationPolicy.numberReplicas = number_replicas
            sysmeta.replicationPolicy.preferredMemberNode = [
                d1_common.types.dataoneTypes.NodeReference("urn:node:mnStageUCSB2")]
        else:
            sysmeta.replicationPolicy.replicationAllowed = False
            sysmeta.replicationPolicy.numberReplicas = 0

        updated_flag = repo.updateSystemMetadata(pid, sysmeta)
        if (updated_flag):
            sysmeta_new = repo.getSystemMetadata(pid)
            view(sysmeta_new, "Updated SystemMetadata")

    except d1_common.types.exceptions.DataONEException as e:
        print(e)

## Update the repository with the new replication policy



In [ ]:
# Try and retrieve existing system metadata
pid_meta = "urn:uuid:c530b4fe-3a14-43b0-8894-dd239b21fc56"
#pid_rmap = "resource_map_urn:uuid:3a06877a-3941-412f-a495-da8608fd4f94"
#pid_png = "urn:uuid:0f93e94d-6509-4c93-8ea1-a6f7129488ef"
change_replication_policy(repo, pid_meta, number_replicas=1)


Original SystemMetadata:
  Identifier: urn:uuid:c530b4fe-3a14-43b0-8894-dd239b21fc56
  Series Identifier: 
  Modified: 2025-07-24 05:27:45.384000+00:00
  Uploaded: 2025-07-24 05:27:44.776000+00:00
  Format ID: https://eml.ecoinformatics.org/eml-2.2.0
  Size: 2476
  Checksum: hash://md5/57860f0890f9dd92a29dd0a7d779cfdb
  Origin Member Node: urn:node:mnTestKNB
  Authoritative Member Node: urn:node:mnTestKNB
  Obsoletes: 
  Obsoleted By: 
  Access policy rules:
    public  can  read
    CN=knb-data-admins,DC=dataone,DC=org  can  read, write, changePermission
  Replication policy:
    Replication allowed: false
    Replicas requested: None
  Replicas of this object:

Updated SystemMetadata:
  Identifier: urn:uuid:c530b4fe-3a14-43b0-8894-dd239b21fc56
  Series Identifier: 
  Modified: 2025-07-24 05:29:52.713000+00:00
  Uploaded: 2025-07-24 05:27:44.776000+00:00
  Format ID: https://eml.ecoinformatics.org/eml-2.2.0
  Size: 2476
  Checksum: hash://md5/57860f0890f9dd92a29dd0a7d779cfdb
  Origin

In [6]:
try:
    pid_meta = "urn:uuid:c530b4fe-3a14-43b0-8894-dd239b21fc56"
    sysmeta2 = cn.getSystemMetadata(pid_meta)
    view(sysmeta2, "Current SystemMetadata")
except d1_common.types.exceptions.DataONEException as e:
    print(e)


Current SystemMetadata:
  Identifier: urn:uuid:c530b4fe-3a14-43b0-8894-dd239b21fc56
  Series Identifier: 
  Modified: 2025-07-24 05:29:52.713000+00:00
  Uploaded: 2025-07-24 05:27:44.776000+00:00
  Format ID: https://eml.ecoinformatics.org/eml-2.2.0
  Size: 2476
  Checksum: hash://md5/57860f0890f9dd92a29dd0a7d779cfdb
  Origin Member Node: urn:node:mnTestKNB
  Authoritative Member Node: urn:node:mnTestKNB
  Obsoletes: 
  Obsoleted By: 
  Access policy rules:
    public  can  read
    CN=knb-data-admins,DC=dataone,DC=org  can  read, write, changePermission
  Replication policy:
    Replication allowed: true
    Replicas requested: 1
    Preferred node: urn:node:mnStageUCSB2
  Replicas of this object:
    urn:node:mnTestKNB completed  2025-07-24 05:29:55.075000+00:00
    urn:node:cnStage completed  2025-07-24 05:29:55.151000+00:00
    urn:node:mnStageUCSB2 completed  2025-07-24 05:30:49.365000+00:00


# Data Packages

Data packages in DataONE are essentially graphs describing the relatonships between objects in a dataset. They are represented as [ORE Resource Maps](https://dataoneorg.github.io/api-documentation/design/DataPackage.html), which are special files that contain metadata about the dataset and its constituent objects.

![Data Package](images/data-package.png)

In order to change the replication policy for an entire dataset, we need to find all of the members of the data package and then change the replication policy for each member. For example, here is a DataONE query executed in `curl` that uses the SOLR index in DataONE to query and find all of the members of a data package:

```bash
❯ curl -s "https://cn-stage.test.dataone.org/cn/v2/query/solr/?q=identifier:urn\:uuid\:c530b4fe-3a14-43b0-8894-dd239b21fc56&fl=resourceMap,documents&wt=json" | jq .response.docs
[
  {
    "documents": [
      "urn:uuid:c530b4fe-3a14-43b0-8894-dd239b21fc56",
      "urn:uuid:c9330b48-ba9a-4a41-8f8a-e8b8cd88650d",
      "urn:uuid:23f752b5-7950-447e-901b-70464bdeee54"
    ],
    "resourceMap": [
      "resource_map_urn:uuid:7a0b9271-4525-4334-845f-6a16602d857b"
    ]
  }
]
```

By iterating over the list of identifiers and calling the `change_replication_policy()` function, we can change the replication policy for each member of the data package. The `change_replication_policy()` function will also display the system metadata before and after the change, so you can see the effect of the change.


# DataONE Replication Summary

![DataONE Replication](./images/dataone-replication.png)


## Manual replication

The replication policy approach above relies on the CN's replication manager to schedule and perform copies. This section outlines an alternative: a client copies (*clones*) a data package, including every version of every object in it, from one Member Node to another using only DataONE API calls, and the CN then recognizes the copies as **replicas** of the originals rather than as new objects. Throughout this section:

- The **source node** is the Member Node that is truly authoritative for the objects being cloned: the node named in `authoritativeMemberNode` in the CN's system metadata for each object. Bytes are downloaded from it, and its system metadata (held on the CN) is what the copies end up with.
- The **target node** is the Member Node that receives the copies. It holds them as replicas and never becomes authoritative for them.

An object whose authoritative node is any node other than the source node is outside the scope of a clone.

This is an outline for a client tool, not yet working code. The behavior described here comes from reading the source code of Metacat (`develop`, `9bfc5065`, for Member Nodes; `2.19.1`, the release the CN runs), the CN synchronization service (`d1_synchronization` 2.3.7, `93d66c9`, as deployed) and the CN REST service (`d1_cn_rest`), and still needs to be confirmed against the staging environment.

The CN and the Member Nodes run very different Metacat implementations, with different storage subsystems: the CN runs Metacat 2.19, and current Member Nodes run Metacat 3, which stores objects in HashStore. CN behavior is described from Metacat 2.19.1. Member Node behavior is described from current Metacat; the parts the process depends on (what `create()` accepts and changes, and the checks in `systemMetadataChanged()`) are the same in Metacat 2.19.1, so the process should also work with a target node that hasn't been upgraded. Where storage makes a difference, both versions are described.

### How DataONE detects replicas across nodes

The CN periodically *harvests* each Member Node by calling `MN.listObjects()` for objects modified since the last harvest ([`ObjectListHarvestTask`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/ObjectListHarvestTask.java#L500)). The harvest does not filter by authoritative node, so a repository lists every object it holds, including copies of objects that belong to other nodes. For each harvested PID, the CN synchronization task ([`V2TransferObjectTask`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/V2TransferObjectTask.java)) decides what the object is:

| CN already knows the PID? | Harvested from the authoritative MN? | Result |
|---|---|---|
| No | (not checked) | Registered as a **new object**, with the harvested node as its only replica ([`processNewObject`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/V2TransferObjectTask.java#L459)) |
| Yes | Yes | Treated as a **system metadata update** from the authoritative node ([`processV2AuthoritativeUpdate`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/V2TransferObjectTask.java#L1314)) |
| Yes | No | If the identifier and checksum match, the node is added as a **replica** with status `completed` ([`processPossibleNewReplica`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/V2TransferObjectTask.java#L1172)) |

In the replica case, the CN compares only the identifier and the checksum recorded in the replica node's system metadata, without reading the object's bytes ([`validateEssentialProperties`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/type/SystemMetadataValidator.java#L138)); if the algorithms differ, it asks the MN for a checksum in the CN's algorithm. The rest of the system metadata on the replica node is ignored: the CN copy, which comes from the authoritative node, stays the master copy.

To record the replica, synchronization calls the CN's `CNReplication.updateReplicationMetadata()` service. On the CN, that service is part of Metacat ([`CNodeService.updateReplicationMetadata`](https://github.com/NCEAS/metacat/blob/2.19.1/src/edu/ucsb/nceas/metacat/dataone/CNodeService.java#L1992)): it adds the replica entry, increments the object's `serialVersion`, and, when the status is `completed`, calls `MN.systemMetadataChanged()` on **every** Member Node in the replica list, including the one just added ([`notifyReplicaNodes`](https://github.com/NCEAS/metacat/blob/2.19.1/src/edu/ucsb/nceas/metacat/dataone/CNodeService.java#L2135)).

Synchronization then sends a second round of notifications of its own ([`V2TransferObjectTask.notifyReplicaNodes`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/V2TransferObjectTask.java#L1431)), which skips the node that was just harvested. So the target is notified once, by Metacat on the CN, and the other replica nodes twice. The two methods have the same name, `notifyReplicaNodes()`, which makes this easy to misread.

A Metacat node that is **not** the authoritative node for an object responds to `systemMetadataChanged()` by fetching the CN copy and storing it in place of its own ([`systemMetadataChanged`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L1836)). That is how the target ends up with the authoritative node's system metadata.

So once an object has been created correctly on the target, the rest happens **automatically**: the CN harvests the target, records the object as a replica, and notifies the target, which stores the CN's system metadata. Two things limit what the automatic path can be relied on for:

- **Each notification is attempted once.** If the target can't be reached or returns an error, the CN logs the failure and moves on ([`notifyReplicaNodes`](https://github.com/NCEAS/metacat/blob/2.19.1/src/edu/ucsb/nceas/metacat/dataone/CNodeService.java#L2135)). Nothing retries it, so the target's copy stays incomplete until the CN changes that object again for some other reason.
- **Some objects can't be refreshed at all**: older versions in a chain of objects that share a series ID (see *Series IDs (SIDs): a known limitation* below).

#### Two modes

The tool described below has two modes:

- **Automatic mode** (the default) needs only the caller's own credentials. The tool creates the objects on the target, then waits for the CN to harvest, register and notify, checking each object along the way. Anything that hasn't happened within a set time is reported, with the likely cause and how to fix it.
- **CN mode** is used when the caller also provides the CN's node certificate **and** asks to act as the CN (`act_as_cn=True`). The tool then makes the CN's calls itself: it registers each replica directly with `CN.updateReplicationMetadata()` (`PUT /v2/replicaMetadata/{pid}`, a public DataONE API call restricted to CN identities) as soon as the object is on the target, without waiting for a harvest, and it repairs missed notifications by calling `systemMetadataChanged()` on the target.

### Conditions that must be met

For each object copied to the target node:

1. **The CN already knows the PID.** Otherwise, the CN registers the target's copy as the original object (first row of the table above). Never clone a PID that the CN can't resolve.
2. **Identical bytes and checksum.** The target must hold exactly the bytes held by the authoritative node, under the same PID. Otherwise synchronization rejects the object, and it never becomes a replica.
3. **`authoritativeMemberNode` stays set to the source node.** Metacat's `create()` keeps a remote `authoritativeMemberNode` value and only fills it in when it is empty ([`create`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L761)). That value decides whether the target later accepts the CN's system metadata ([`isAuthoritativeNode`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L3877)). If the target believes it is authoritative, it never takes the CN's copy, and the field can't be changed afterwards.
4. **Use `create()` only, never `update()`.** On a node that isn't authoritative for an object, Metacat only allows `update()` and `updateSystemMetadata()` from a CN administrator ([`doUpdateAuth`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/D1AuthHelper.java#L215)), so replaying a version history with `create()` followed by `update()` fails. Instead, every version is created on its own, and the version chain is restored afterwards from the CN. (The subsection on series IDs below discusses one possible exception, using the CN's credentials.)
5. **Clear the version-chain fields before `create()`.** `create()` rejects system metadata that has `obsoletes` or `obsoletedBy` set, or a series ID (SID) already used on the target.
6. **The target is registered in the CN's node registry as a version 2 Member Node.** The CN only notifies replica nodes it finds in the node registry. In automatic mode, the target must also have **synchronization enabled**, or the CN never harvests it.
7. **In CN mode, the tool checks what synchronization would have checked.** Registering a replica directly skips the CN's own comparison of checksums, so the tool must confirm that the target holds the right bytes before registering it.

### Overview of the process

1. **Discover**: find every PID in every version of the package, using the CN and the source node.
2. **Plan**: get the system metadata for each PID from the CN, build a version graph, check it for gaps, decide which PIDs can be cloned, and choose an order of creation.
3. **Create**: for each PID, download the bytes from the source and `create()` the object on the target, with system metadata stripped of the version-chain fields.
4. **Registration as a replica**:
   - *automatic mode*: wait until the CN has harvested the target and lists it as a `completed` replica;
   - *CN mode*: confirm the target's checksum, then register the replica directly with `CN.updateReplicationMetadata()`.

   Either way, the CN then notifies the target, which stores the CN's system metadata. This restores `obsoletes`, `obsoletedBy`, SIDs, the origin node, and the dates.
5. **Refresh**:
   - *automatic mode*: wait for the target's copy to be updated by the CN's notification;
   - *CN mode*: if the target's copy wasn't updated, call `systemMetadataChanged()` on the target.
6. **Verify and report**: compare the target's system metadata with the CN's, and report every PID that didn't end up correct, with the likely cause.

Steps 1 and 2 work on the package as a whole. Steps 3–6 are done for each PID separately, and each PID can move on as soon as its own condition is met:

- Creates don't depend on each other, so they don't have to wait for the CN between versions.
- A PID can become a replica as soon as it's on the target (CN mode) or harvested (automatic mode), without waiting for the rest of the package.
- Each refresh is independent, because the CN copy of every version carries both its `obsoletes` and its `obsoletedBy` pointers. The version chains are complete once every PID has been refreshed, whatever the order.

The tool keeps track of where each PID is with a small state machine, described under *Orchestration* below. That lets it handle large packages, run creates in parallel, wait for the CN without blocking, and resume after an interruption.

The sections below outline a client tool, `clone()`, that implements this process. The caller is assumed to hold credentials for the target node (to create objects) and credentials that can read the objects on the source (for any private objects). The CN's node certificate is optional and only used in CN mode.

### Step 0: Clients, credentials and mode

Set up clients for the CN, the source node and the target node with the caller's credentials. If the caller provides the CN's certificate and sets `act_as_cn=True`, also set up the CN and the target node with the CN's certificate, and use CN mode.

```python
def make_context(cn_url, source_url, target_url, user_token,
                 cn_cert=None, cn_key=None, act_as_cn=False):
    headers = {"Authorization": "Bearer " + user_token}
    ctx = Context()
    ctx.cn     = CoordinatingNodeClient_2_0(cn_url, headers=headers)
    ctx.source = MemberNodeClient_2_0(source_url, headers=headers)   # must be able to read private objects
    ctx.target = MemberNodeClient_2_0(target_url, headers=headers)   # must be allowed to create()
    ctx.source_node_id = ctx.source.getCapabilities().identifier.value()
    ctx.target_node_id = ctx.target.getCapabilities().identifier.value()

    node = find_node(ctx.cn.listNodes(), ctx.target_node_id)
    if node is None or node.type != "mn":
        raise Setup("the target isn't a Member Node in the CN's node registry")

    if act_as_cn:
        if not (cn_cert and cn_key):
            raise Setup("act_as_cn=True needs the CN's certificate and key")
        ctx.mode = "cn"
        ctx.cn_as_cn = CoordinatingNodeClient_2_0(cn_url, cert_pem_path=cn_cert, cert_key_path=cn_key)
        ctx.target_as_cn = MemberNodeClient_2_0(target_url, cert_pem_path=cn_cert, cert_key_path=cn_key)
    else:
        if cn_cert:
            warn("a CN certificate was given without act_as_cn=True; it won't be used")
        if not node.synchronize:
            raise Setup("automatic mode needs synchronization enabled for the target node")
        ctx.sync_schedule = QuartzSchedule.from_node(node)   # when the CN harvests the target (Step 4)
        ctx.mode = "automatic"
    return ctx
```

The certificate alone doesn't switch on CN mode: the caller has to ask for it, so that the CN's identity is never used just because a certificate happens to be configured.

**Pitfalls**
- The CN credential has to be an identity that Metacat treats as a CN: a subject listed in the `subjectList` of a CN entry in the node registry ([`doCNOnlyAuthorization`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/D1AuthHelper.java#L298)). A user account that happens to be an administrator won't work. Keep this certificate out of the notebook and the repository.
- In CN mode the certificate is used to **write** to the CN (Step 4), which is a more powerful use than notifying the target: it can change the replica records of any object.
- The target may limit who can submit objects (submitter allow list, quota service), and the caller's token may not be enough.
- Use the CN and nodes from the same environment (for example, `cn-stage` with staging repositories).

### Step 1: Discover every PID in every version of the package

Start from any PID in the package, find its resource map, and then gather the full set of PIDs: every version of the resource map, every member of each of those versions, and every version of each member.

```python
def collect_package_pids(cn, source, package_pid):
    rm_head = find_resource_map(cn, package_pid)        # package_pid itself, or the Solr `resourceMap` field
    rm_versions = walk_version_chain(cn, rm_head)       # follow obsoletes / obsoletedBy both ways
    seeds = set(rm_versions)
    for rm in rm_versions:
        seeds |= members_of(source, rm)                 # parse the ORE document; recurse into nested resource maps
    return version_closure(cn, seeds)                   # add every version of every seed, until nothing new appears
                                                        # returns (pids, unreachable) as in walk_version_chain

def walk_version_chain(cn, pid):
    chain, unreachable, todo = set(), {}, [pid]
    while todo:
        p = todo.pop()
        if p in chain or p in unreachable: continue
        try:
            sm = cn.getSystemMetadata(p)
        except (NotAuthorized, NotFound) as e:
            unreachable[p] = e                          # a gap: record it and keep walking the other branches
            continue
        chain.add(p)
        todo += [x.value() for x in (sm.obsoletes, sm.obsoletedBy) if x is not None]
    return chain, unreachable
```

A version that the caller can't read, or that the CN no longer has, must not stop the walk: it's recorded as *unreachable*, and Step 2 reports it as a gap in the version graph.

**Pitfalls**
- Get the members of each resource-map version from the ORE document itself, not only from Solr: search results can leave out archived objects, and the index may be incomplete or stale for older versions.
- Nested packages: a resource map can list child resource maps among its members. Recurse into them, or explicitly decide not to.
- Shared objects: a data object can belong to several packages. Following its version chain may pull in versions that belong to other packages. That's harmless, but should be reported.
- A resource map can list PIDs that don't exist or that the caller can't read. Add them to the unreachable set rather than failing quietly.
- The CN's `getSystemMetadata()` enforces read access, so versions the caller isn't allowed to read show up as `NotAuthorized`. This is the most common cause of gaps.
- The source node can hold versions that never synchronized to the CN. Compare the source's list with the CN's (Step 2), because only PIDs known to the CN can be cloned.

### Step 2: Build the version graph and the plan

Get the CN's system metadata for each PID, classify each PID, and build a graph whose nodes are PIDs and whose edges are the `obsoletes` relationships. Check the graph for gaps, then decide the order of creation and which PID keeps each series ID.

```python
def build_plan(cn, source, target, pids, source_node_id):
    plan = {}
    for pid in pids:
        cn_sm = try_get(cn.getSystemMetadata, pid)          # None if NotFound
        if cn_sm is None:
            plan[pid] = Skip("not on CN: fix synchronization on the source first"); continue
        if cn_sm.authoritativeMemberNode.value() != source_node_id:
            plan[pid] = Skip("authoritative node is " + cn_sm.authoritativeMemberNode.value()); continue
        target_sm = try_get(target.getSystemMetadata, pid)
        if target_sm is not None:
            plan[pid] = AlreadyPresent(checksum_ok=same_checksum(target_sm, cn_sm)); continue
        plan[pid] = Create(cn_sm, format_type=cn.getFormat(cn_sm.formatId).formatType)

    sid_heads = {sid: cn.getSystemMetadata(sid).identifier.value()      # a SID resolves to its head version
                 for sid in {p.sysmeta.seriesId.value() for p in creates(plan) if p.sysmeta.seriesId}}

    rank = {"DATA": 0, "METADATA": 1, "RESOURCE": 2}
    order = sorted(creates(plan), key=lambda p: (rank[p.format_type], p.sysmeta.dateUploaded))
    return plan, sid_heads, order
```

#### Check the version graph for gaps

A version chain should be a single connected line: each PID's `obsoletedBy` points to a PID whose `obsoletes` points back, from the first version to the head. A **gap** is anywhere that isn't true for a PID in the package:

| Gap | How it shows up | Usual cause |
|---|---|---|
| Unreadable version | a pointer leads to a PID that returns `NotAuthorized` | the caller doesn't have read permission on that version |
| Missing version | a pointer leads to a PID that returns `NotFound` | the version was deleted, or never synchronized to the CN |
| Broken link | `A.obsoletedBy = B`, but `B.obsoletes` isn't `A` (or the reverse) | inconsistent system metadata on the source node |
| Excluded version | the PID is readable, but the plan skips it | not on the CN, or another node is authoritative |
| Branch | two PIDs both claim to obsolete the same PID | inconsistent system metadata on the source node |

```python
def check_version_graph(sysmeta_by_pid, unreachable, plan):
    gaps = []
    for pid, sm in sysmeta_by_pid.items():
        for field, back in [("obsoletes", "obsoletedBy"), ("obsoletedBy", "obsoletes")]:
            other = getattr(sm, field)
            if other is None: continue
            other = other.value()
            if other in unreachable:
                gaps.append(Gap(pid, field, other, kind=classify(unreachable[other])))   # unreadable / missing
            elif other in sysmeta_by_pid and value(getattr(sysmeta_by_pid[other], back)) != pid:
                gaps.append(Gap(pid, field, other, kind="broken link"))
            elif isinstance(plan.get(other), Skip):
                gaps.append(Gap(pid, field, other, kind="excluded: " + plan[other].reason))
    gaps += find_branches(sysmeta_by_pid)          # more than one PID obsoleting the same PID
    return group_by_chain(gaps)                    # report per object: which chain, where the gap is
```

Report the gaps before anything is created, grouped by chain, so the user can decide whether to go ahead. Cloning across a gap is possible but leaves a lasting effect on the target node: after Step 5, the versions on either side of the gap carry `obsoletes` or `obsoletedBy` pointers (from the CN's system metadata) to PIDs that don't exist on the target node, so the version history there can't be followed past the gap. Gaps caused by permissions can usually be closed by running the discovery and plan again with an identity that can read every version, such as the rights holder's. The CN certificate could read them too, but it's safer to keep its use to CN mode's calls in Steps 4 and 5.

#### Order of creation

All data objects come first, then metadata, then resource maps, with older versions before newer ones within each type. On the target, `create()` calls don't depend on each other, so this order isn't needed for correctness. It keeps the target's search index from seeing a resource map before its members exist, and makes an interrupted run easier to reason about.

**Pitfalls**
- **PIDs the CN doesn't know** must be left out (condition 1). Cloning one would make the target the authoritative copy on the CN.
- **PIDs already on the target** should be checked: a matching checksum means a previous run got that far, so skip the PID (this lets the tool resume). A different checksum means stop: the PID is already in use on the target for a different object.
- **Another authoritative node:** a PID whose authoritative node isn't the source node is outside the scope of the clone. Leave it out; the gap check reports it as an excluded version if it's part of a version chain.
- **Replication policy:** the CN's replica registration doesn't check the object's replication policy, so it accepts a copy even on a node listed in `blockedMemberNode` or when `replicationAllowed` is false. The tool should check the policy itself and refuse, or require an explicit override.
- **Gaps** in the version graph (see above) don't stop the plan by default. `clone()` refuses to go on unless the caller passes `allow_gaps=True`, so the decision is explicit.
- Print the plan, including any gaps, as a dry run before creating anything.

### Step 3: Prepare the system metadata and create each object

For each PID in the plan, copy the CN's system metadata, clear the fields that `create()` rejects or that the target shouldn't claim, download the bytes from the source, check them against the checksum, and create the object on the target.

```python
def prepare_sysmeta(cn_sm, sid_heads):
    sm = copy.deepcopy(cn_sm)
    sm.replica = []                     # the CN maintains the replica list
    sm.obsoletes = None                 # create() rejects version-chain fields;
    sm.obsoletedBy = None               #   the refresh in Step 5 restores them
    pid = sm.identifier.value()
    if sm.seriesId is not None and sid_heads[sm.seriesId.value()] != pid:
        sm.seriesId = None              # only one PID per node can hold a SID
    # keep authoritativeMemberNode = source node (essential, see condition 3)
    # keep checksum, size, formatId, rightsHolder, accessPolicy, replicationPolicy
    return sm

def create_on_target(source, target, pid, sm):
    path = download_to_tempfile(source.get(pid, stream=True))
    if checksum(path, sm.checksum.algorithm) != sm.checksum.value():
        raise ChecksumMismatch(pid)
    with open(path, "rb") as f:
        target.create(pid, f, sm)
    # the target's own timestamp, used in automatic mode to tell when the harvest has picked it up (Step 4)
    return target.getSystemMetadata(pid).dateSysMetadataModified
```

The orchestrator (below) calls `create_on_target()` for each PID in the `PLANNED` state, several at a time, and keeps to the creation order from Step 2.

**Pitfalls**
- **Fields that `create()` overwrites:** `originMemberNode`, `submitter`, `dateUploaded`, `dateSysMetadataModified`, `serialVersion` (set to 0) and `archived` (set to `false`) all take local values ([`create`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L761)). Step 5 restores them. Until then, archived versions are visible as unarchived on the target, and older versions appear as separate objects that aren't linked to each other.
- **DOIs:** `create()` registers DOI metadata when the PID starts with a DOI shoulder that the target is configured for, which could point the DOI at the target. Check the target's DOI settings before cloning DOI-identified objects (staging repositories may share the `10.5072/FK2` test shoulder).
- **Large objects:** stream them to disk instead of reading them into memory, and allow for long upload times.
- **Who checks the bytes:** Metacat on the target computes the checksum of the upload and rejects the object if it doesn't match the system metadata. The two Member Node versions do this in different storage code: Metacat 3 while the upload streams into HashStore ([`StreamingMultipartRequestResolver`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/restservice/multipart/StreamingMultipartRequestResolver.java#L175)), and Metacat 2.19 while writing the object to its file store ([`writeStreamToFile`](https://github.com/NCEAS/metacat/blob/2.19.1/src/edu/ucsb/nceas/metacat/dataone/D1NodeService.java#L1846)). The CN never reads the bytes of a replica: it only compares the checksum recorded on the target with its own. (Its setting `Synchronization.checksum.verify.size.bypass.threshold`, 10,000,000 bytes in production, only applies to new objects, not to replicas.) The tool's own check before uploading is still worth doing: it catches a bad download before it's sent, and says clearly which side the problem is on.
- **Access:** private objects need credentials on the source that can read them. The access policy is copied as-is, so the target applies the same rules.
- **Quotas:** the target's quota service counts each create against the caller.
- **`IdentifierNotUnique`:** a previous run may have created the object before stopping. Compare the target's checksum with the CN's: if they match, treat the PID as created; if not, stop and investigate.

### Step 4: Registration as a replica

#### Automatic mode: wait for the CN to harvest the target

The CN's synchronization service registers each replica itself when it next harvests the target. There's no need to check a PID before the harvest has picked it up. The node registry records, for each node, `lastHarvested`: the `dateSysMetadataModified` of the newest object queued for synchronization in the last harvest, not the time the harvest ran ([`ObjectListHarvestTask`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/ObjectListHarvestTask.java#L196)). Once it is at or after the timestamp the target gave a PID when it was created (Step 3), the PID is in the CN's synchronization queue. From then on, check the PID's replica entry on the CN until it's `completed`, or until a deadline passes.

#### When to expect the harvest

The node registry also records **when** the CN harvests each node. In the node list (`GET /cn/v2/node`, or `cn.listNodes()`), each node's `synchronization` element has a `schedule` whose attributes are the fields of a [Quartz Scheduler](https://www.quartz-scheduler.org/documentation/quartz-2.3.0/tutorials/crontrigger.html) cron expression, in Quartz's order:

```xml
<synchronization>
    <schedule sec="10" min="0/3" hour="*" mday="*" mon="*" wday="?" year="*"/>
    <lastHarvested>2026-11-05T00:03:17.776+00:00</lastHarvested>
    <lastCompleteHarvest>1900-01-01T00:00:00.000+00:00</lastCompleteHarvest>
</synchronization>
```

Joined in that order, the attributes make the expression `10 0/3 * * * ? *`: every 3 minutes, 10 seconds past the minute. On the staging CN, most Member Nodes are harvested every 3 minutes, and some once a day (for example `0 9 9 * * ? *`, at 09:09:00).

Together, the two values answer different questions: the schedule says **when** it's worth looking, and `lastHarvested` says **whether** a harvest has picked the object up. `lastHarvested` doesn't record when the last harvest ran: it only moves when a harvest finds newer objects (on staging, `urn:node:mnStageUCSB2` is harvested every 3 minutes, but its `lastHarvested` was 2026-08-21), so it can't be used on its own to tell whether harvests are running.

#### How much one harvest takes

A harvest doesn't necessarily pick up everything that's new on the target. Each run takes at most

`min(Synchronization.max_harvest_size, Synchronization.max_syncobjectqueue_size − length of the CN's synchronization queue)`

objects ([`ObjectListHarvestTask`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/ObjectListHarvestTask.java#L144)). In production both are set to the defaults in the code: 2,000 objects per harvest, and 4,000 for the queue. The queue length is the total for **all** Member Nodes (`SyncQueueFacade.size()` adds up the queues kept for each node), even though the setting's comment in the configuration describes it as a limit for each node's queue. Within that limit, the harvest takes the **oldest** objects first, by `dateSysMetadataModified`, and only stops between timestamps, never in the middle of one ([`SortedHarvestTimepointMap`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/SortedHarvestTimepointMap.java)). It then sets `lastHarvested` to the timestamp of the last object it queued, so the next run continues from there.

So objects that don't fit are picked up by later runs, in order, and the per-PID check (`lastHarvested` at or after the PID's timestamp) stays correct however many runs it takes. What changes is how long it can take. With a schedule of every few minutes and a modest number of objects, one run usually takes everything. A large clone on a node harvested daily or weekly can need many runs: at 2,000 objects per run, 25,000 new objects need at least 13 harvests, and more if the target has other new objects ahead of them, or if other nodes' work fills the queue. If the CN's synchronization queue is already full (with work from any node), a run harvests nothing at all.

#### Waiting for the harvest

The tool uses the schedule and `lastHarvested` together:

- **Don't check before the next harvest.** A PID waiting for its harvest is next checked shortly after the next time the schedule fires, not at a fixed interval. For a node harvested once a day, that saves a day of pointless checks; for one harvested every 3 minutes, it keeps the checks frequent.
- **Wait as long as the harvest is making progress.** While PIDs are waiting, the tool keeps track of `lastHarvested` for the target. As long as it moves forward, the harvester is working through the target's objects, and the tool keeps waiting, however many runs that takes. Only if `lastHarvested` hasn't moved over several scheduled harvests (three by default) while objects are waiting does the tool report the waiting PIDs as `harvest stalled`. A deadline in hours, or in a fixed number of harvests, would give up too early on a large clone, and wait far too long on a node whose synchronization has stopped.
- **Estimate how long it will take.** The number of objects on the target that are ahead of a PID in the harvest is `target.listObjects(fromDate=lastHarvested, toDate=<the PID's timestamp>, count=0).total`. Divided by the per-run limit, that gives the number of harvests still needed, and with the schedule, a time to show in progress reports. The per-run limit isn't published by the CN, so it's an argument to the tool, set by default to the production value of 2,000. In practice the estimate is a lower bound, because a run takes fewer objects when the CN's queue is busy.

```python
class QuartzSchedule:
    FIELDS = ("sec", "min", "hour", "mday", "mon", "wday", "year")   # Quartz order

    @classmethod
    def from_node(cls, node):
        s = node.synchronization.schedule
        return cls(" ".join(getattr(s, f) for f in cls.FIELDS))      # e.g. "10 0/3 * * * ? *"

    def fires_after(self, t, n):
        ...                                    # the next n times the schedule fires after t

    def fires_between(self, t0, t1):
        ...                                    # how many times the schedule fired in (t0, t1]

class HarvestProgress:                         # one per target node, kept in the database
    def observe(self, cn, target_node_id):     # once per pass of the orchestrator
        last = find_node(cn.listNodes(), target_node_id).synchronization.lastHarvested
        if last != self.last_harvested:
            self.last_harvested, self.changed_at = last, now()

    def stalled(self, schedule, deadlines):
        return schedule.fires_between(self.changed_at, now() - deadlines.after_harvest) \
               >= deadlines.stalled_fires

    def runs_ahead(self, target, target_modified, harvest_limit):   # for progress reports
        ahead = target.listObjects(fromDate=self.last_harvested, toDate=target_modified, count=0).total
        return ceil(ahead / harvest_limit)

def replica_status(sm, node_id):
    return {r.replicaMemberNode.value(): r.replicationStatus for r in sm.replica}.get(node_id)

def check_harvested(progress, target_modified, schedule, deadlines):
    if progress.last_harvested >= target_modified:
        return Ok()                            # queued for synchronization on the CN
    if progress.stalled(schedule, deadlines):
        return Failed("harvest stalled")
    return Wait(until=schedule.fires_after(now(), 1)[0] + deadlines.after_harvest)

def check_registered(cn, pid, target_node_id, harvested_at, deadlines):
    if replica_status(cn.getSystemMetadata(pid), target_node_id) == "completed":
        return Ok()
    if now() > harvested_at + deadlines.registration:
        return Failed("not registered")
    return Wait(until=now() + backoff_while_queued(pid))
```

Compare the target's own timestamps (both values come from the target's clock), not the time on the client. Being harvested only means being queued: the CN's synchronization queue can still delay a PID, which is why the replica entry is checked separately after the harvest, with its own deadline (`deadlines.registration`), counted from when the tool first saw the PID harvested.

**Pitfalls (automatic mode)**
- **How long this takes** depends on the target's synchronization schedule, the per-run harvest limit, the number of new objects on the target, and how busy the CN's synchronization queue is. Waiting for the harvest has no fixed deadline, only the check for a stalled harvest; the registration deadline after the harvest is a fixed time and should be set from what's normal for the CN. A deadline that's too short reports PIDs that would have been registered later. Reported PIDs are still checked again on the next run.
- **`lastHarvested` can stand still for a good reason:** a run harvests nothing when the CN's synchronization queue is full, and a single timestamp shared by more objects than the limit is harvested in one larger run. A stall reported after a few harvests may just mean the CN is busy; running the tool again later picks the PIDs up again.
- **Refreshed copies drop out of later harvests.** A refresh gives the target's copy the CN's `dateSysMetadataModified`, which is older than the time the copy was created. That's harmless: by then the CN has registered the replica, and a later harvest that saw the object again would find it already registered.
- **Quartz expressions aren't standard cron expressions:**
  - they start with a seconds field and can end with a year field;
  - one of the two day fields must be `?` ("no specific value");
  - days of the week run from 1 (Sunday) to 7 (Saturday), not 0–6;
  - `0/3` means "every 3, starting at 0";
  - `L`, `W` and `#` have special meanings in the day fields.

  Python cron libraries such as `croniter` use different field orders and day numbering, so don't pass the expression to them unchanged. A small parser for the forms in use (`*`, `?`, numbers, lists, ranges and `/` steps) is enough. For anything it doesn't understand, the tool should fall back to fixed deadlines and say so.
- **Time zone:** the expression is evaluated by the CN's scheduler, in the CN's time zone. That's assumed here to be UTC; confirm it for the CN being used. A margin of some minutes also covers differences between the target's clock and the CN's.
- **A scheduled harvest can be late.** A harvest that's still running when the next one is due delays it, and a harvest of many objects can take a long time. That's why a harvest only counts as stalled after several firings without progress, not one.
- **The schedule can change.** Read it from the node registry at the start of every run, not once.
- **`harvest stalled`** usually means synchronization is turned off or stuck for the target, or that the CN's synchronization queue has been full for a while. Check the node registry entry and the CN's synchronization status.
- **`not registered`** after the harvest usually means synchronization rejected the object, for example because of a checksum mismatch or a PID the CN doesn't know. The CN reports it to the target with `MN.synchronizationFailed()`, so the target's Metacat log has the reason.
- Each check of a replica entry is one CN call. With thousands of PIDs waiting, check each PID less often the longer it waits (see *Orchestration*).

There is no way for a client to *push* a replica to the CN instead. `CN.synchronize(pid)` ([`CoreController.synchronize`](https://github.com/DataONEorg/d1_cn_rest/blob/0609c11/src/main/java/org/dataone/cn/rest/v2/CoreController.java#L359)) only accepts PIDs the CN doesn't yet know, or calls from the object's authoritative node, and returns `NotAuthorized 4692` otherwise. Called by the source node, it would only make the CN look for changes on the source.

#### CN mode: register directly

With the CN's certificate, record the target as a `completed` replica of each created PID by calling `CN.updateReplicationMetadata()`, the same call the CN's synchronization service makes when it finds a replica. This doesn't wait for a harvest. Because synchronization isn't involved, the tool first checks what synchronization would have checked: that the target holds an object with the right checksum and size.

```python
from d1_common.types import dataoneTypes

def register_one(cn, cn_as_cn, target, pid, target_node_id):
    sm = cn.getSystemMetadata(pid)
    if replica_status(sm, target_node_id) == "completed":
        return Ok("already registered")            # for example, by a harvest of the target
    t_sm = target.getSystemMetadata(pid)
    t_sum = target.getChecksum(pid, sm.checksum.algorithm)
    if t_sum.value() != sm.checksum.value() or t_sm.size != sm.size:
        return Failed("checksum", (t_sum.value(), t_sm.size))
    replica = dataoneTypes.Replica()
    replica.replicaMemberNode = target_node_id
    replica.replicationStatus = "completed"
    replica.replicaVerified = utc_now()
    try:
        cn_as_cn.updateReplicationMetadata(pid, replica, int(sm.serialVersion))
        return Ok()                                # the CN has also notified the target (see Step 5)
    except VersionMismatch:
        return Retry()                             # the CN changed the object meanwhile: read it again
    except (ServiceFailure, Timeout):
        return Retry()                             # the call may have succeeded: the retry looks first
```

`updateReplicationMetadata()` is `PUT /v2/replicaMetadata/{pid}` on the CN, handled by Metacat ([`updateReplicationMetadata`](https://github.com/NCEAS/metacat/blob/2.19.1/src/edu/ucsb/nceas/metacat/restservice/v2/CNResourceHandler.java#L1804)). d1_python's `CoordinatingNodeClient_2_0.updateReplicationMetadata()` sends it in the form Metacat expects (a `replicaMetadata` XML part and a `serialVersion` field). Replication status values are lowercase in the DataONE schema (`completed`, `requested`, `failed`, ...).

**Pitfalls (CN mode)**
- **The checksum and size checks are the only check.** Synchronization normally compares checksums before recording a replica; registering directly skips that, so never register a PID without them.
- **`serialVersion` must be current.** The CN rejects the call with `VersionMismatch` if the object changed after it was read. Read the system metadata again and retry. A harvest of the target can register the replica at the same moment, which also shows up as a `VersionMismatch`; the retry then finds the replica already registered.
- **Existing replica entries for the target.** The CN refuses to change a `completed` entry to anything except `invalidated`, but it lets a `requested`, `queued` or `failed` entry, for example from an earlier attempt by the replication manager, be replaced by `completed`. The replication manager may still have a task queued for that entry. Step 2 should report such entries, so that the tasks can be checked before cloning.
- **The call notifies the replica nodes before it returns.** The CN calls `systemMetadataChanged()` on every Member Node holding a replica of the object, within the same request, so the call can take a while, and the client may time out even though the CN finished. That's why a timeout leads to a retry that looks before acting.
- **Load on the CN and on other replica nodes.** Every registration makes the CN notify every node holding the object. For large packages, limit how many registrations are made per minute.

### Step 5: Refresh of the target's system metadata

However the replica was registered (Step 4), the CN then calls `systemMetadataChanged()` on the target. Because the target's copy names the source as authoritative, Metacat fetches the CN's system metadata and stores it in place of its own copy ([`systemMetadataChanged`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L1836)). This restores the version chains (`obsoletes`, `obsoletedBy`), SIDs, the origin node, the submitter, the dates, the archived flag, and the replica list.

The CN only attempts that notification once. Step 6 checks each PID, and finds copies that weren't updated (*stale* copies). What happens to them depends on the mode:

- **Automatic mode:** the tool keeps checking for a while, in case the CN changes the object again, then reports the PID with the likely cause (see the failure report under *Orchestration*). The usual cause, a missed notification, can be repaired later by running the tool in CN mode, or by making a change to the object on the source (see below).
- **CN mode:** the tool calls `systemMetadataChanged()` on the target itself, with the CN's certificate.

```python
def refresh_one(cn, target_as_cn, pid):          # CN mode only
    sm = cn.getSystemMetadata(pid)
    try:
        target_as_cn.systemMetadataChanged(pid, int(sm.serialVersion), sm.dateSysMetadataModified)
        return Ok()
    except InvalidRequest as e:
        if e.detailCode == "1334" and sm.seriesId is not None:
            return Failed("sid", e)        # known limitation: see the subsection on series IDs
        return Failed("refresh", e)
    except DataONEException as e:
        return Failed("refresh", e)
```

**Pitfalls**
- **Series IDs:** if the target's copy has no SID and the CN's SID is already held by another PID on the target, Metacat rejects the refresh with `InvalidRequest 1334` and stores nothing. Since only the head version keeps its SID in Step 3, **every older version in a SID chain is never refreshed**, whichever way the notification arrives. Chains without SIDs are restored fully. See *Series IDs (SIDs): a known limitation* below.
- **In automatic mode the SID error isn't visible**: it only appears in the CN's log. The tool can only tell that the PID is in a SID chain and stays stale, and reports it as *probably SID*. In CN mode, the tool's own refresh returns the `1334` error.
- Metacat only accepts the refresh if the local `serialVersion` is no greater than the CN's. `create()` sets it to 0, so this always holds for new copies.
- **Repairing a missed notification without the CN certificate:** make a real change to the object's system metadata on the source, for example in the replication policy (as `change_replication_policy()` does). The CN then synchronizes that change and notifies every replica node except the source, including the target. This modifies the source's system metadata, needs write permission on the source, and the change has to count as a valid update to the CN, which a date-only change may not. The tool shouldn't do this on its own; the report suggests it.

### Step 6: Verify the clone

Once the target's copy has been refreshed (Step 5), the target node holds a copy of the CN's system metadata as it was at that moment ([`systemMetadataChanged`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L1836) stores the CN copy whole), so every field should match the CN's, not just a chosen few. Compare the entire record, using the comparison in `d1_common.system_metadata`, which normalizes both copies first: it sorts the access policy, the replica list, and other lists where order has no meaning, and rounds timestamps. That covers fields like `dateUploaded`, `submitter`, `originMemberNode`, `fileName`, `mediaType` and the replication policy, all of which `create()` overwrote or might have dropped.

`serialVersion` decides how to read a difference. The CN increments it whenever it changes an object's system metadata, for example when it registers another replica:

- If the target's `serialVersion` equals the CN's, the two copies should be **identical**, including the replica list. Any difference is a problem.
- If the target's `serialVersion` is lower, the target hasn't received the CN's latest copy. The target's copy is **stale**, not wrong. Step 5 explains what happens next in each mode. The likely cause tells the user what to do:
  - the target's `serialVersion` is still 0, the value `create()` gave it: the target was never refreshed, so the notification was missed;
  - the CN's copy has a SID and the target's doesn't: *probably SID*, which no refresh can fix;
  - otherwise, the target was refreshed once, and the CN has changed the object since.

```python
import copy
import d1_common.system_metadata as d1_sm

def verify_one(cn, target, pid, target_node_id):
    cn_sm, t_sm = cn.getSystemMetadata(pid), target.getSystemMetadata(pid)
    if t_sm.serialVersion < cn_sm.serialVersion:
        return Stale(stale_cause(cn_sm, t_sm))         # Step 5 decides what happens next
    problems = []
    a, b = copy.deepcopy(cn_sm), copy.deepcopy(t_sm)   # the comparison normalizes in place
    if not d1_sm.are_equivalent_pyxb(a, b):
        problems.append(field_differences(a, b))       # per-field report, for people to read
    if target_node_id not in [r.replicaMemberNode.value() for r in cn_sm.replica]:
        problems.append("CN does not list the target node as a replica")
    if target.getChecksum(pid, cn_sm.checksum.algorithm).value() != cn_sm.checksum.value():
        problems.append("stored bytes")
    return Failed("verify", problems) if problems else Ok()

def stale_cause(cn_sm, t_sm):
    if cn_sm.seriesId is not None and t_sm.seriesId is None:
        return "probably sid"
    if t_sm.serialVersion == 0:
        return "notification missed"
    return "changed since refresh"

def verify_package(target, pids):           # once every PID has finished
    check_sids_resolve(target, pids)        # target.get(sid) should return the head version
    check_package_indexed(target, pids)     # the target's search index shows the package and its members
```

**Pitfalls**
- `are_equivalent_pyxb()` changes both of its arguments while normalizing them, so pass it copies.
- Don't use `d1_sm.clear_elements()` to ignore the replica list: it also clears the replication policy, which hides a field we do want to compare.
- A comparison that only says "not equivalent" isn't enough to act on. `field_differences()` should name each field that differs, so that, for example, an older version in a SID chain (Step 5) is reported as *probably SID* rather than as a dozen separate differences.
- If tests on staging show that the replica list legitimately differs even when `serialVersion` matches, exclude the replica list from the comparison. Don't loosen anything else.
- The target's search index updates asynchronously after Step 5, so the index checks may need to be retried.

### Orchestration: a state machine for each PID

Steps 1 and 2 run once per call of `clone()`, for the whole package. From then on, each PID moves through a series of states on its own, and moves to the next state only when its condition is met. The states are kept in a small SQLite database, so that a run can be stopped and started again at any point, and so that the progress of a large package can be followed.

| State | Meaning | Automatic mode | CN mode | Next state |
|---|---|---|---|---|
| `PLANNED` | in the plan, not yet on the target | `create_on_target()` (Step 3), once its type is allowed to start | same | `CREATING` |
| `CREATING` | a create is in progress | | | `CREATED`; back to `PLANNED` if the run was interrupted |
| `CREATED` | on the target, not yet a replica on the CN | `check_harvested()` after each scheduled harvest, until a harvest has picked it up, or the harvest has stalled (Step 4) | `register_one()` (Step 4) | `HARVESTED` (automatic), `REGISTERED` (CN mode) |
| `HARVESTED` | queued for synchronization on the CN (automatic mode only) | `check_registered()` until it's registered, or the registration deadline passes (Step 4) | | `REGISTERED` |
| `REGISTERED` | a replica on the CN; the target should have been notified | `verify_one()` (Step 6) | same | `VERIFIED` or `STALE` |
| `STALE` | the target's copy isn't the CN's latest | `verify_one()` again later, until the refresh deadline passes | `refresh_one()` (Step 5) | `VERIFIED` (automatic), `REGISTERED` (CN mode) |
| `VERIFIED` | done | | | |
| `SKIPPED` | left out by the plan, with the reason (Step 2) | | | |
| `FAILED` | something went wrong, with the reason and the number of attempts | reported | retried with increasing delays, up to a limit, where a retry can help | |

```sql
CREATE TABLE clone_pid (
    target_node     TEXT NOT NULL,
    pid             TEXT NOT NULL,
    package_pid     TEXT NOT NULL,   -- the package that first brought this PID into a clone
    format_type     TEXT NOT NULL,   -- DATA, METADATA or RESOURCE
    date_uploaded   TEXT NOT NULL,   -- for the creation order
    target_modified TEXT,            -- from create_on_target(); tells when a harvest has picked it up
    state           TEXT NOT NULL,
    reason          TEXT,            -- for SKIPPED and FAILED, and the cause for STALE
    attempts        INTEGER NOT NULL DEFAULT 0,
    refreshes       INTEGER NOT NULL DEFAULT 0,   -- STALE -> REGISTERED cycles in CN mode
    state_since     TEXT NOT NULL,   -- when it entered its current state, for the deadlines
                                     -- (for HARVESTED: when the tool first saw it harvested)
    next_check      TEXT,            -- when to look at this PID again
    PRIMARY KEY (target_node, pid)   -- a PID shared by several packages is cloned once
);

CREATE TABLE harvest_progress (      -- automatic mode: is the CN getting through the target's objects?
    target_node    TEXT PRIMARY KEY,
    last_harvested TEXT NOT NULL,    -- the node registry's lastHarvested, as last seen
    changed_at     TEXT NOT NULL     -- when the tool last saw it change
);
```

The orchestrator is one loop. Each pass moves every PID that is ready by one state, then waits a little before the next pass:

```python
def reconcile_once(db, ctx):
    moved = 0
    # Step 3: create, a few at a time, one format type after another
    for ftype in ("DATA", "METADATA", "RESOURCE"):
        ready = db.pids(state="PLANNED", format_type=ftype, due=True)
        moved += run_in_pool(ctx.pool, lambda pid: create_step(db, ctx, pid), ready)
        if db.count(state=["PLANNED", "CREATING"], format_type=ftype) > 0:
            break                          # don't start the next type until this one is created

    # Step 4: registration as a replica
    if ctx.mode == "cn":
        for pid in db.pids(state="CREATED", due=True):
            result = register_one(ctx.cn, ctx.cn_as_cn, ctx.target, pid, ctx.target_node_id)
            ctx.rate_limit.wait()          # each registration makes the CN notify every replica node
            moved += db.record(pid, result, next_state="REGISTERED")
    else:
        progress = db.harvest_progress(ctx.target_node_id)
        progress.observe(ctx.cn, ctx.target_node_id)     # one node-registry call per pass
        for pid in db.pids(state="CREATED", due=True):
            result = check_harvested(progress, db.target_modified(pid), ctx.sync_schedule, ctx.deadlines)
            moved += db.record(pid, result, next_state="HARVESTED")   # Wait(until=...) sets next_check
        for pid in db.pids(state="HARVESTED", due=True):
            result = check_registered(ctx.cn, pid, ctx.target_node_id, db.state_since(pid), ctx.deadlines)
            moved += db.record(pid, result, next_state="REGISTERED")

    # Step 6: verify
    for pid in db.pids(state=["REGISTERED", "STALE"], due=True):
        result = verify_one(ctx.cn, ctx.target, pid, ctx.target_node_id)
        if isinstance(result, Stale) and ctx.mode == "automatic" and db.in_state(pid, "STALE") \
                and now() - db.state_since(pid) > ctx.deadlines.refresh:
            result = Failed(result.cause)  # report it: notification missed, probably sid, ...
        moved += db.record(pid, result, next_state="VERIFIED", if_stale="STALE")

    # Step 5, CN mode only: refresh what verification found out of date
    if ctx.mode == "cn":
        for pid in db.pids(state="STALE"):
            if db.refreshes(pid) >= 3:
                db.fail(pid, "keeps going stale"); continue
            moved += db.record(pid, refresh_one(ctx.cn, ctx.target_as_cn, pid),
                               next_state="REGISTERED", count_refresh=True)
    return moved

def create_step(db, ctx, pid):
    db.move(pid, "CREATING")               # so that a crash in the middle is noticed on restart
    if already_on_target_with_same_checksum(ctx.target, ctx.cn, pid):
        return db.move(pid, "CREATED", target_modified=ctx.target.getSystemMetadata(pid).dateSysMetadataModified)
    sm = prepare_sysmeta(ctx.cn.getSystemMetadata(pid), db.sid_heads())
    db.record(pid, attempt(create_on_target, ctx.source, ctx.target, pid, sm), next_state="CREATED")

def clone(package_pid, ctx, db, dry_run=True, allow_gaps=False,
          harvest_limit=2000,                           # Synchronization.max_harvest_size in production; for estimates only
          deadlines=Deadlines(stalled_fires=3,          # scheduled harvests without progress (automatic mode)
                              after_harvest=minutes(1), # how soon after a scheduled harvest to look
                              registration=hours(6),    # CN's synchronization queue, after the harvest
                              refresh=hours(1))):
    pids, unreachable = collect_package_pids(ctx.cn, ctx.source, package_pid)          # Step 1
    plan, sid_heads, order = build_plan(ctx.cn, ctx.source, ctx.target, pids, ctx.source_node_id)  # Step 2
    gaps = check_version_graph(sysmeta_of(plan), unreachable, plan)
    report(plan, sid_heads, order, gaps, mode=ctx.mode)
    if dry_run: return plan
    if gaps and not allow_gaps: raise VersionGraphHasGaps(gaps)

    ctx.deadlines = deadlines
    with db.lock(ctx.target_node_id):      # one run per target node at a time
        db.load_plan(package_pid, plan, sid_heads)   # adds new PIDs; never moves existing ones backwards
        db.reset_interrupted("CREATING", to="PLANNED")
        db.recheck_failed(package_pid)     # a new run looks again at PIDs reported last time
        while not db.all_finished(package_pid):
            moved = reconcile_once(db, ctx)
            sleep(seconds(10) if moved else minutes(1 if ctx.mode == "cn" else 5))
    verify_package(ctx.target, db.pids(package_pid, state="VERIFIED"))
    return failure_report(db, package_pid)
```

Running `clone()` again with the same package is always safe: the plan is rebuilt, PIDs already in the database keep their state, every action first checks what is actually on the target and the CN, and PIDs reported as failed are looked at again. A PID reported in automatic mode because its harvest was late is picked up by the next run once the CN has caught up.

#### The failure report

`clone()` ends with a report of every PID that isn't `VERIFIED`, grouped by reason:

| Reason | What it means | What to do |
|---|---|---|
| `skipped: ...` | left out by the plan (not on the CN, another authoritative node, policy) | fix the cause on the source, or accept it |
| `harvest stalled` | the CN's harvests of the target stopped making progress before reaching the object (automatic mode) | check that synchronization is enabled and running for the target, and whether the CN's synchronization queue is full; run again later, or in CN mode |
| `not registered` | harvested, but the CN didn't record the replica (automatic mode) | look for `synchronizationFailed` in the target's Metacat log, usually a checksum problem |
| `checksum` | the target's bytes don't match the CN's checksum | delete the object on the target and clone it again |
| `notification missed` | registered, but the target was never refreshed | run again in CN mode, or make a change to the object on the source (Step 5) |
| `changed since refresh` | the CN changed the object after the refresh, and the target missed that notification | as for `notification missed` |
| `probably sid` / `sid` | an older version in a SID chain (Step 5) | see *Series IDs (SIDs): a known limitation* |
| `verify` | the copies have the same `serialVersion` but differ | investigate: the report lists the fields |

With direct registration (CN mode), the time a clone takes is mostly the time spent uploading. In automatic mode, add the time until the CN's scheduled harvests of the target have reached every new object, which can be one harvest or many, minutes apart or a day apart depending on the node, and the time the CN takes to work through its synchronization queue. How the same design handles different package sizes, assuming each object has a few versions:

| Members | PIDs (approx.) | Settings |
|---|---|---|
| 10 | 30–50 | One worker for creates. |
| 100 | 300–500 | One or two workers. |
| 1,000 | 3,000–5,000 | A few workers. Uploads take hours, so expect to stop and resume runs. |
| 5,000 | 15,000–25,000+ | A few workers at most, to protect the target's storage and indexing. In CN mode, also limit registrations per minute, to protect the CN and the other replica nodes. In automatic mode, this many objects need many harvest runs (at least 8–13 at 2,000 objects per run), which on a daily schedule means one to two weeks or more; the progress reports show the estimate. |

**Pitfalls**
- **Creation order with parallel creates:** creating several objects at once can put a resource map on the target before its members. The loop above waits until every PID of one format type has been created before starting the next. A resource map whose member has `FAILED` should wait too, unless the user decides otherwise.
- **Start with few workers.** Each create makes the target node store and index the object. Increase the number of workers only while the target keeps up.
- **Deadlines in automatic mode** decide when the tool stops waiting and reports. They don't change what the CN does, so a PID reported too early is simply checked again on the next run. Waiting for the harvest has no fixed deadline: the tool waits as long as the harvests are making progress, so it suits large clones and nodes harvested once a day as well as small clones on nodes harvested every few minutes.
- **The database can drift from reality**, for example if someone deletes an object on the target, or a harvest registers a replica before CN mode does. That's why each action checks the target and the CN first, rather than trusting the database.
- **One run per target node:** two runs creating on the same target node at once would create objects twice and confuse each other's states. Hold a lock for the whole run.
- **The CN certificate** is only needed by `register_one()` and `refresh_one()`. Load it only for those clients, and only in CN mode.

### Series IDs (SIDs): a known limitation

The process above doesn't fully restore older versions of objects that have a series ID. This subsection explains why and lists the possible solutions. Which one to use is still to be decided.

#### What happens

Take a series `S` with three versions, `v1 → v2 → v3`, on the source node. All three carry `seriesId = S`. The source node allows that because each version was made with `update()`, which accepts a SID that matches the previous version's.

1. **On `create()`:** Metacat's `create()` rejects a SID that's already used on the node ([`create`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L761)). So only one of the three versions can carry `S` on the target. The plan (Step 2) gives it to the head, `v3`, and removes it from `v1` and `v2`.
2. **On the refresh:** for `v1`, the target's copy has no SID, and the CN's copy has `S`. Metacat's `systemMetadataChanged()` takes the branch "no current SID, so the new SID must not exist yet". `S` already exists, on `v3`, so Metacat throws `InvalidRequest 1334` **before storing anything** ([`systemMetadataChanged`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/MNodeService.java#L1836)). `v1` keeps the system metadata it got from `create()`: no `obsoletes` or `obsoletedBy`, the target as origin node, the cloner as submitter, and the clone's dates. The same happens to `v2`.
3. **No order of operations avoids it.** Whichever version holds `S` first blocks the others. If `S` is removed from every version, the first one refreshed gets it, and the rest fail.
4. **It keeps failing later.** Every future notification from the CN for `v1` or `v2` fails the same way. For example, if the source node later archives `v1`, the archive never reaches the target.

The bytes on the target are correct, and the CN still holds the right system metadata. Only the target node's local copy is wrong, and only for older versions in SID chains.

#### Why normal replication doesn't have this problem

When the CN's replication manager copies an object, it calls `MN.replicate()` on the target. That method skips `MNodeService.create()` altogether (its code says "skip the MN.create -- this mutates the system metadata") and stores the CN's system metadata whole, with the SID, `obsoletes` and origin node intact. The process above goes through the client-facing `create()`, which enforces rules meant for brand-new objects.

#### Possible solutions

1. **Change Metacat (preferred).** As far as we can tell, the SID check in `systemMetadataChanged()` serves no purpose in either branch of the version 2 API:
   - on the authoritative node, the CN's copy is thrown away anyway, and only the replica list and `serialVersion` are kept;
   - on a replica node, the CN is the authority, and it has already checked the SID chain during synchronization ([`validateSeriesId`](https://github.com/DataONEorg/d1_synchronization/blob/93d66c9/src/main/java/org/dataone/cn/batch/synchronization/tasks/V2TransferObjectTask.java)).

   Skipping the check when `needCheckAuthoriativeNode` is true would let the refresh accept SIDs the same way `replicate()` does. Once a fixed Metacat is deployed on the target, refreshing the affected PIDs again repairs existing clones: in CN mode, the tool's refresh does it; otherwise, the next notification from the CN for each PID does, which may require a change to the object on the source.
2. **Replay SID chains with `update()`, using the CN's credentials.** `doUpdateAuth` accepts CN administrators even on a node that isn't authoritative ([`doUpdateAuth`](https://github.com/NCEAS/metacat/blob/9bfc5065/src/edu/ucsb/nceas/metacat/dataone/D1AuthHelper.java#L215)). Create `v1` with `S`, then `update(v1 → v2)` and `update(v2 → v3)`, which carries `S` through the chain because `update()` accepts a SID that matches the previous version's. The refresh then succeeds, because the target's SID already matches the CN's. The costs: each SID chain has to be created strictly in order, one version at a time, and the CN certificate is used for more than the refresh. Not yet checked: whether `update()` on a node that isn't authoritative contacts the CN in a way that causes trouble.
3. **Drive `replicate()` directly, using the CN's credentials.** This keeps the CN's system metadata whole. However, the source node only serves the object to the target through `getReplica()` if `CN.isNodeAuthorized()` approves, which depends on replication state recorded on the CN. That state would have to be created through the CN's replication API, which brings the replication manager back in. This is the riskiest option.
4. **Accept the limitation.** Clone anyway, and report the affected PIDs (`FAILED` with reason `sid`) so that they can be repaired later with option 1 or 2.

In the state machine, PIDs affected by this limitation end in `FAILED` with reason `sid` (CN mode, where the tool sees the error) or `probably sid` (automatic mode, where the error only appears in the CN's log), and aren't retried automatically.

### Open issues and limitations

- **Older versions in a SID chain** aren't fully restored. See *Series IDs (SIDs): a known limitation* above.
- **Gaps in the version graph** can be cloned across with `allow_gaps=True`, but the target node's version history can't be followed past them.
- **PIDs the CN doesn't know** can't be cloned this way. They have to be synchronized from the source first.
- **CN mode writes to the CN with the CN's certificate.** Whether that's acceptable is a policy decision, which is why it's optional and has to be asked for explicitly. Automatic mode doesn't need it, but is slower and can only report missed notifications, not repair them.
- **Automatic mode depends on the target's synchronization.** If it's turned off or stuck, nothing is registered.
- **This process bypasses the replication policy and the CN's replication manager.** The new copies count toward `numberReplicas`, and the tool is responsible for respecting `replicationAllowed` and `blockedMemberNode`.
- None of this has been tested against the staging environment yet.

## The `datamgmt` R client: existing clone functions

The [`datamgmt`](https://github.com/NCEAS/datamgmt) R package already has a module of clone functions, [`clone_package.R`](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/clone_package.R). This section evaluates them against the process described above, and lists the changes needed to make them as reliable. It's based on reading `datamgmt` (`main`, `ac799c5`) and the packages it relies on: `arcticdatautils` (`develop`, `fc9740f`), `dataone` (`rdataone`, `92d2f9f`) and `datapack` (`c865aa5`).

### Summary

The functions were written mainly to **copy** packages: to give them new PIDs and make the target node authoritative, typically to edit a package or restore an old version. With the right arguments they can also create objects on the target in the way a replica clone needs (Step 3 above), but only for the **current version** of a package. They don't do the other steps: discovering every version, the plan, waiting for registration, refreshing, verifying, or resuming. And one bug makes the replica-style combination of arguments fail every time.

### The functions, one by one

#### [`clone_object()`](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/clone_package.R#L46)

Copies one object. Called with `new_pid = FALSE, change_auth_node = FALSE, add_access_to = NULL, public = FALSE`, it creates a copy that the CN can recognize as a replica: it keeps the PID and the `authoritativeMemberNode` (`dataone::createObject()` only fills in fields that are empty, see [`createObject`](https://github.com/DataONEorg/rdataone/blob/92d2f9f/R/MNode.R#L370)), and it clears `obsoletes` and `obsoletedBy`, as Metacat's `create()` requires.

Gaps, compared with Step 3:

- **The defaults are for copying.** `new_pid` defaults to `TRUE`, and `add_access_to` has no default, so a replica clone has to pass every argument explicitly.
- **`add_access_to` and `public = TRUE` change the access policy on the target.** For a replica, this is only harmless if the target's copy is later refreshed. If the notification is missed, the target keeps the broader access, and with `public = TRUE`, a private object becomes public there.
- **`seriesId` isn't cleared**, so the second version of a SID chain fails on the target with `InvalidSystemMetadata 1180`. The replica list isn't cleared either (`datapack` keeps it in a [`replica` slot](https://github.com/ropensci/datapack/blob/c865aa5/R/SystemMetadata.R#L64); whether it's written back out wasn't checked).
- **System metadata comes from the `from` Member Node, not the CN** ([`getDataObject`](https://github.com/DataONEorg/rdataone/blob/92d2f9f/R/D1Client.R#L282) calls `getSystemMetadata(x@mn, ...)`), so for a replica clone `from` has to be the authoritative node.
- **The whole object is held in memory** (`getDataObject(limit = "1TB")` returns the bytes), and the downloaded bytes aren't checked against the checksum before the upload.
- **Every error becomes a warning and a `NULL` result** ([lines 112–122](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/clone_package.R#L112-L122)). The caller can't tell an object that's already on the target (`IdentifierNotUnique`, which a resumed run expects) from a real failure.
- **No checks** that the CN knows the PID, or that its authoritative node is the source.

#### [`clone_package()`](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/clone_package.R#L176)

Finds the members of one resource map and calls `clone_object()` for each of them, then for the resource map. With `new_pid = FALSE` it clones the resource map object as it is, which is right for a replica, and `clone_children = TRUE` recurses into nested packages.

Gaps, compared with Steps 1–6:

- **Only the current version.** The members come from [`arcticdatautils::get_package()`](https://github.com/NCEAS/arcticdatautils/blob/fc9740f/R/util.R#L696), a search-index query (`resourceMap:<pid>`, at most 5,000 rows) on the `from` node, falling back to the CN. It finds the members of that one resource map: there's no walk of the version history, no version graph and no check for gaps.
- **Objects with an unknown `formatType` are silently left out:** [`get_package_direct()`](https://github.com/NCEAS/arcticdatautils/blob/fc9740f/R/util.R#L744) collects them, but doesn't return them.
- **Failed data objects are silently dropped** (`unlist()` removes the `NULL` results), so with `new_pid = FALSE` the cloned resource map lists members that aren't on the target.
- **The replica combination of arguments always fails at the end.** The last step calls [`arcticdatautils::set_access()`](https://github.com/NCEAS/arcticdatautils/blob/fc9740f/R/access.R#L174) with `subjects = add_access_to` ([line 306](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/clone_package.R#L306)), and `set_access()` stops when `subjects` is `NULL`. So a replica clone, which needs `add_access_to = NULL`, ends with an error after every object has been created. When subjects are given, the step is usually a no-op, because `clone_object()` already added those rules; when it isn't, it calls `updateSystemMetadata()` on a node that isn't authoritative, which Metacat refuses.
- **More than one metadata object breaks it.** `clone_object()`'s argument check, `if (!(is.character(pid) & nchar(pid) > 0))`, fails on a vector (an error since R 4.2).
- **Nothing happens after the creates:** no waiting for the CN to register the replicas, no refresh, no comparison of system metadata, no report, and no record of progress for resuming.

The order of creation (metadata, data, child packages, then the resource map) is fine: the resource map comes last, which is the part that matters.

#### [`copy_package()`](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/clone_package.R#L354)

A true copy: new PIDs, the target as the authoritative node, and access for the caller. That's a different operation from a replica clone and is fine as it is; it shouldn't share arguments with a replica clone.

#### [`arcticdatautils::get_all_versions()`](https://github.com/NCEAS/arcticdatautils/blob/fc9740f/R/util.R#L625) and [`query_all_versions()`](https://github.com/NCEAS/datamgmt/blob/ac799c5/R/query_all_versions.R)

`get_all_versions()` walks `obsoletes` backwards and `obsoletedBy` forwards with `getSystemMetadata()`, and warns and stops when a link can't be followed. It's a good starting point for Step 1, but:

- it treats every error the same way, so it can't tell a missing version (`NotFound`) from one the caller can't read (`NotAuthorized`);
- it doesn't check that the links agree in both directions (broken links, branches);
- it walks whichever node it's given; for a clone, that should be the CN.

`query_all_versions()` adds a search-index query for each version, which is useful for reports. It reads the list of index fields from a hard-coded Arctic Data Center address.

#### [Tests](https://github.com/NCEAS/datamgmt/blob/ac799c5/tests/testthat/test_clone_package.R)

All the tests clone within a single node (`urn:node:mnTestARCTIC` to itself), mostly with `new_pid = TRUE`. None clones with `new_pid = FALSE` between two nodes, the replica case, which is why the `set_access()` failure hasn't been noticed.

#### The R `dataone` client and CN mode

`rdataone` can authenticate with X.509 certificates as well as tokens, but it has no methods for the CN's replication API (`updateReplicationMetadata()`) or for `MN.systemMetadataChanged()`. CN mode in R would need those added to `rdataone`, or direct HTTP calls. Whether `rdataone`'s certificate support can be pointed at a CN node certificate hasn't been checked.

### Recommended changes to the R client

To be as reliable as the process described above, the R client would need:

1. **A separate function for replica clones**, rather than another combination of `clone_package()` arguments. Copying and replicating need different defaults and different checks. `copy_package()` stays as it is, for copies.
2. **Discovery (Step 1):** collect every version of every member and of the resource map, with a version of `get_all_versions()` that walks the CN, records unreachable versions with the reason (`NotFound` or `NotAuthorized`), and keeps going on other branches. Read the members of each resource-map version from the ORE document, not only the search index, and include objects with an unknown `formatType`.
3. **The plan (Step 2):** skip PIDs the CN doesn't know, or whose authoritative node isn't the source; detect PIDs already on the target, comparing checksums; check the version graph for gaps and refuse to continue without an explicit `allow_gaps`; check the replication policy; report existing replica entries for the target; check DOI shoulders; choose which PID keeps each SID; and offer a dry run.
4. **Creating (Step 3):** start from the CN's system metadata; clear `obsoletes`, `obsoletedBy`, the replica list, and SIDs on versions other than the head; never change the access policy or `authoritativeMemberNode`; stream large objects to disk and check their checksums before uploading; treat `IdentifierNotUnique` with a matching checksum as already done; return a typed result for each PID instead of a warning; and drop the final `set_access()` call.
5. **Waiting and checking (Steps 4–6):** in automatic mode, wait for each PID using `lastHarvested` and then its replica entry on the CN, with deadlines; compare the target's system metadata with the CN's in full (R has no equivalent of `d1_common.system_metadata.are_equivalent_pyxb()`, so this needs writing); and classify copies that weren't refreshed as *notification missed*, *probably SID* or *changed since refresh*.
6. **CN mode (optional):** `updateReplicationMetadata()` and `systemMetadataChanged()` with the CN's certificate, which first need adding to `rdataone`.
7. **Orchestration:** a record of each PID's state so that runs can be resumed, parallel creates within each format type, polling that slows down while nothing changes, and a final failure report.
8. **Tests:** clone between two staging nodes with the PIDs kept, including a package with several versions and one with a SID chain.

Two of these are bugs that affect the existing functions today, whatever is decided about the rest: the `set_access()` call that fails when `add_access_to` is `NULL`, and the argument check that fails for packages with more than one metadata object.

### How the R client relates to the Python design

The two can share the same design: the conditions, the steps and the failure reasons above apply to both. What differs is where each piece fits best.

- **Most of the work is outside what `datamgmt` does now.** `clone_object()` covers part of Step 3 for one version, and `get_all_versions()` covers part of Step 1. Steps 2 and 4–6, CN mode, and the state machine would all be new.
- **The orchestration fits Python better.** Running for hours, keeping state in SQLite, uploading in parallel and polling the CN are a natural fit for a stand-alone Python tool built on `d1_python`, which already has the system metadata comparison, the CN replication call, and certificate authentication. In R, each of those would have to be added first.
- **A suggested split:** build the reliable clone tool in Python, following the design above. In `datamgmt`, fix the two bugs, and either add a small replica-clone function for single packages that follows the rules of Steps 2 and 3 and then reports what to check, or point R users at the Python tool. `copy_package()` stays for copying.